[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Performance and Memory


## What you will be able to do

Measure what each column of a table costs, shrink it with the right types, read a file a piece at a
time, and choose the fastest of three ways to compute a new column from your own measurements.

## The idea

### The problem

A year of hourly readings at four stations is 35,040 rows, and the table's three columns take 8
bytes a row each, about 840,000 bytes in all. The same network with 400 stations and ten years of
history is 35 million rows, and about 840 million bytes at the same cost.

A third of that is the station names, four words repeated 35,040 times, with every row holding an
8-byte reference to a Python string, as much room as a column of decimals. A file read with the
defaults adds more. Every column pandas cannot place becomes text, the times included, and a column
you never use is still loaded.

Speed has the same shape. A loop over the rows of a table runs Python code once for every row, and
35 million rows is 35 million turns of that loop.

### What a dtype and a category cost

> A column's **dtype** fixes how each of its values is stored. A `float64` or `int64` value takes 8
> bytes, a `float32` 4 and an `int8` 1, while an `object` column holds an 8-byte reference for every
> row to a Python object stored elsewhere, and rows with the same value often share one object. A
> **category** column stores each distinct value once and keeps a small integer code for every row.

### Why it works that way

`memory_usage(deep=True)` adds each row's Python string to its reference, and counts a shared string
again for every row that refers to it, so it reports what the column would take if no two rows
shared a string. Here four strings sit behind 35,040 names, so the station column really takes
little more than its 280,320 bytes of references. Without `deep=True`, pandas counts only the
references, and `info` marks such a total with a `+`.

A category fits a column with few distinct values and many rows. Four station names become four
strings and 35,040 one-byte codes, an eighth of the references they replace. The hour of the day
fits in a smaller number type, `int8`, which holds -128 to 127. `float32` halves a column of
decimals and keeps about seven significant digits, so -3.6 is stored as -3.5999999046325684, and the
difference shows when the column is compared with the original.

`read_csv` decides the types while it reads, and `usecols` and `dtype` let you decide first, so the
large version of a column never exists. `chunksize` returns the file as a series of smaller tables,
so only one piece is in memory at a time.

A vectorized expression runs a compiled loop over the column for each operation, as in the **NumPy,
Deep Dive** guide. `apply` calls a Python function for every value, and `iterrows` builds a Series
for every row.

### Where this shows up

This matters for tables built from logs, sensors or transactions, with millions of rows and columns
of names and codes. The **Performance** notebook in the **NumPy, Deep Dive** guide measured how much
faster an array is than a loop, and the same measurement holds for a column. The **Reading Data**
notebook used `usecols` and `dtype` to read a file correctly, and here they make it small.

### What this notebook covers

- what each column of `readings` costs
- the station names as a category, and the decimals and hours as smaller numbers
- a file read with only the columns it needs, in the types it needs
- a file read a piece at a time
- a loop, `apply` and one vectorized expression, timed
- four errors, among them a `float32` column that no longer matches the readings it came from

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things.

```python
import pandas as pd

names = pd.Series(["Bergen", "Oslo"] * 50_000)
print(names.memory_usage(deep=True))
print(names.astype("category").memory_usage(deep=True))
```

```
5400132
100348
```

`memory_usage(deep=True)` counted 5.4 million bytes for a hundred thousand names as text and 100,348
as a category. The list repeats two string objects, so the text column really holds 800,000 bytes of
references, and the category, which stores the two names once and a one-byte code for each row, is
about an eighth of that.

## Setup

One library, seven helpers, and the readings twice, as a table and as a file.

- `pandas`, imported as `pd`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings
- `Path` names the scratch folder and the file in it, and `shutil` removes the folder at the end
- `timeit` times the three ways of computing a column, in the last worked example
- `warnings` catches the warning that grouping by a category gives, in Common errors

`readings` and `stations` are the tables of the **Why Pandas** notebook. `readings.to_csv` writes
the readings to `scratch/readings.csv` without the index, so that the worked examples can read a
file back, and the cell prints the file's size.

In [1]:
import math
import shutil
import timeit
import warnings
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
CSV = SCRATCH / "readings.csv"
readings.to_csv(CSV, index=False)                                    # the readings as a file, to read back

print("pandas", pd.__version__, "|", len(readings), "readings, and", f"{CSV.stat().st_size:,}", "bytes in", CSV)

pandas 2.2.3 | 35040 readings, and 1,109,641 bytes in scratch/readings.csv


## Worked examples

### What each column costs

`memory_usage` returns the bytes each column takes, with the index as its own entry:

In [2]:
print(readings.memory_usage().to_dict(), "| total:", f"{readings.memory_usage().sum():,}")
print(readings.memory_usage(deep=True).to_dict())
print("total, deep:", f"{readings.memory_usage(deep=True).sum():,}", "bytes")

{'Index': 132, 'station': 280320, 'time': 280320, 'celsius': 280320} | total: 841,092
{'Index': 132, 'station': 1927200, 'time': 280320, 'celsius': 280320}
total, deep: 2,487,972 bytes


`time` and `celsius` take 280,320 bytes each, 8 bytes for each of 35,040 rows, either way. `station`
takes the same 280,320 without `deep=True`, which counts only its references, and 1,927,200 with it,
because `deep=True` adds a 47-byte string to every row's 8-byte reference. The next cell shows the
rows share four strings, so the column really takes close to its 280,320 bytes:

In [3]:
print(len({id(name) for name in readings["station"]}), "string objects behind", len(readings), "station names")

4 string objects behind 35040 station names


`id` is the identity of a Python object, so a set of the ids counts the distinct strings. `info`
gives the same two counts as `memory_usage`. Its default skips the strings and marks the total with
a `+`:

In [4]:
readings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35040 entries, 0 to 35039
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   station  35040 non-null  object        
 1   time     35040 non-null  datetime64[ns]
 2   celsius  35016 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 821.4+ KB


In [5]:
readings.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35040 entries, 0 to 35039
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   station  35040 non-null  object        
 1   time     35040 non-null  datetime64[ns]
 2   celsius  35016 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 2.4 MB


### The station names as a category

`astype("category")` keeps each distinct name once and a code for every row:

In [6]:
compact = readings.astype({"station": "category"})

print(compact["station"].cat.categories.tolist())
print(compact["station"].cat.codes.head(4).tolist(), compact["station"].cat.codes.dtype)
print(compact.memory_usage(deep=True).to_dict())
print("total, deep:", f"{compact.memory_usage(deep=True).sum():,}", "bytes")

['Bergen', 'Oslo', 'Svalbard', 'Tromso']
[0, 1, 2, 3] int8
{'Index': 132, 'station': 35432, 'time': 280320, 'celsius': 280320}
total, deep: 596,204 bytes


The codes are 0 to 3, one byte each, and the column fell from 280,320 bytes of references, 1,927,200
by the deep count, to 35,432. The whole table now takes 596,204 bytes, against 841,092 for the
original's references and numbers. Grouping works as before, so each station's mean is unchanged:

In [7]:
compact.groupby("station", observed=True)["celsius"].mean().round(2)

station
Bergen      8.00
Oslo        6.50
Svalbard   -4.48
Tromso      3.50
Name: celsius, dtype: float64

`observed=True` asks for the categories that appear in the data, and Common errors shows why it is
worth writing.

### Smaller numbers

`pd.to_numeric` with `downcast` picks the smallest type that holds every value. The hour of the day
arrives as `int32` and fits in `int8`, and the decimals fit in `float32`:

In [8]:
hour = readings["time"].dt.hour
small_hour = pd.to_numeric(hour, downcast="integer")
small_celsius = pd.to_numeric(readings["celsius"], downcast="float")

print(hour.dtype, "->", small_hour.dtype, "|", hour.memory_usage(), "->", small_hour.memory_usage(), "bytes")
print(readings["celsius"].dtype, "->", small_celsius.dtype, "|", readings["celsius"].memory_usage(), "->",
      small_celsius.memory_usage(), "bytes")

int32 -> int8 | 140292 -> 35172 bytes
float64 -> float32 | 280452 -> 140292 bytes


Every count includes the 132 bytes of the index. `float32` halves the column and keeps about seven
digits, which is enough for readings to a tenth of a degree, as long as the column is not compared
with the `float64` original, which Common errors shows.

### Reading only what you need

Read with the defaults, the file comes back with `time` and `station` both as text:

In [9]:
everything = pd.read_csv(CSV)

print(everything.dtypes.to_dict())
print(f"{everything.memory_usage(deep=True).sum():,}", "bytes by the deep count")
station_strings = len({id(name) for name in everything["station"]})
time_strings = len({id(time) for time in everything["time"]})
print(f"{station_strings} station strings and {time_strings} time strings behind {len(everything)} rows")

{'station': dtype('O'), 'time': dtype('O'), 'celsius': dtype('float64')}
4,590,372 bytes by the deep count
4 station strings and 8760 time strings behind 35040 rows


`read_csv` shared one string among the rows that repeat a value, four for the stations and one for
each of the 8,760 hours, so the deep count of 4,590,372 bytes overstates the table, which really
holds mostly references and the hours' 8,760 strings. `usecols` reads only the columns named, and
`dtype` gives each its type as it is read:

In [10]:
needed = pd.read_csv(CSV, usecols=["station", "celsius"], dtype={"station": "category", "celsius": "float32"})

print(needed.dtypes.to_dict())
print(f"{needed.memory_usage(deep=True).sum():,}", "bytes")

{'station': CategoricalDtype(categories=['Bergen', 'Oslo', 'Svalbard', 'Tromso'], ordered=False, categories_dtype=object), 'celsius': dtype('float32')}
175,552 bytes


The same file, for a question about stations and temperatures, takes 175,552 bytes, with no
references left and no strings beyond the four category names.

### A file read a piece at a time

`chunksize` makes `read_csv` return the file as a series of tables of that many rows, one at a time.
A mean cannot be averaged across pieces, but a sum and a count can be added up, and divided at the
end:

In [11]:
sums = pd.Series(dtype=float)
counts = pd.Series(dtype=int)
pieces = 0

for piece in pd.read_csv(CSV, chunksize=10_000):
    by_station = piece.groupby("station")["celsius"]
    sums = sums.add(by_station.sum(), fill_value=0)
    counts = counts.add(by_station.count(), fill_value=0)
    pieces += 1

print(pieces, "pieces, the last of", len(piece), "rows")
(sums / counts).round(2)

4 pieces, the last of 5040 rows


station
Bergen      8.00
Oslo        6.50
Svalbard   -4.48
Tromso      3.50
dtype: float64

The four pieces gave the same means as the whole table. `add` with `fill_value=0` lines up the
stations by label and counts a station missing from either side as 0, which is what lets the first
piece go into the empty `sums`. Only one piece was in memory at a time. This file is about a
megabyte and would fit anyway, and the same loop works on a file too large for the computer's
memory, which this notebook does not make.

### Timing a loop, apply and one expression

Every reading can go into Fahrenheit by a loop over the rows with `iterrows`, by `apply` with a
function, or by the formula on the whole column. The cell below checks first that all of them give
the same answer:

In [12]:
def with_iterrows():
    return pd.Series([row["celsius"] * 9 / 5 + 32 for _, row in readings.iterrows()])


def with_apply():
    return readings["celsius"].apply(lambda celsius: celsius * 9 / 5 + 32)


def vectorized():
    return readings["celsius"] * 9 / 5 + 32


print(with_iterrows().equals(vectorized()), with_apply().equals(vectorized()))

True True


`timeit.repeat` runs each one several times, and `min` keeps the fastest run. The seconds depend on
the machine, so the cell prints only the comparisons:

In [13]:
iterrows_time = min(timeit.repeat(with_iterrows, number=1, repeat=3))
apply_time = min(timeit.repeat(with_apply, number=3, repeat=5)) / 3
vectorized_time = min(timeit.repeat(vectorized, number=3, repeat=5)) / 3

print("iterrows took more than 10 times as long as apply:", iterrows_time > 10 * apply_time)
print("apply took more than 8 times as long as the expression:", apply_time > 8 * vectorized_time)

iterrows took more than 10 times as long as apply: True
apply took more than 8 times as long as the expression: True


Over twenty runs on the machine these outputs come from, `iterrows` took between 43 and 169 times as
long as `apply`, and `apply` between 25 and 167 times as long as the expression. Each margin is
below a third of the smallest of those ratios, so another machine is likely to print the same.
`iterrows` builds a Series for every row, `apply` calls a Python function for every value, and the
expression runs a compiled loop over the column for each operation. Reach for the expression first,
`apply` for work no expression can do, and a loop over the rows only for work that has to see one
row at a time, with `itertuples`, which Common errors shows.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/18-performance-and-memory-solutions.ipynb).

**1.** Print the memory of `readings["time"]` with and without `deep=True`, and say why the two agree.

In [14]:
# your code here

**2.** Make `station` a category, and print how many bytes the column takes with `deep=True`, and its categories.

In [15]:
# your code here

**3.** Read only `station` and `celsius` from `CSV`, with `station` as a category, and print the table's memory with `deep=True`.

In [16]:
# your code here

**4.** Read `CSV` in pieces of 5,000 rows, and count the readings below zero across all the pieces.

In [17]:
# your code here

**5.** Convert every reading to kelvin, by adding 273.15, with `apply` and with one expression, check that the two agree, and print whether `apply` took more than 10 times as long.

In [18]:
# your code here

## Common errors

### TypeError: Cannot setitem on a Categorical with a new category (Narvik), set the categories first

In [19]:
station_names = compact["station"].copy()

station_names[0] = "Narvik"

TypeError: Cannot setitem on a Categorical with a new category (Narvik), set the categories first

A category column accepts only the values in its list of categories, and Narvik is not one of them.
Add it to the list first, and the assignment works:

In [20]:
station_names = station_names.cat.add_categories(["Narvik"])
station_names[0] = "Narvik"

print(station_names.cat.categories.tolist(), station_names[0])

['Bergen', 'Oslo', 'Svalbard', 'Tromso', 'Narvik'] Narvik


### FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas.

In [21]:
every_station = pd.Categorical(readings["station"], categories=list(LATITUDES))    # Kirkenes included

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    means_by_category = readings.groupby(every_station)["celsius"].mean().round(2)

for warning in caught:
    print(f"{warning.category.__name__}: {str(warning.message).split('. ')[0]}.")
means_by_category

Bergen      8.00
Oslo        6.50
Svalbard   -4.48
Tromso      3.50
Kirkenes     NaN
Name: celsius, dtype: float64

The category's list includes Kirkenes, which has no readings, and by default pandas 2.2 groups by
every category in the list, so Kirkenes appears with a mean of `NaN`. A future version of pandas
will show only the categories that appear, and the warning says so. `observed=True` gives that
answer now and silences the warning, and `observed=False` keeps the empty groups on purpose:

In [22]:
readings.groupby(every_station, observed=True)["celsius"].mean().round(2)

Bergen      8.00
Oslo        6.50
Svalbard   -4.48
Tromso      3.50
Name: celsius, dtype: float64

### TypeError: 'numpy.float64' object cannot be interpreted as an integer

In [23]:
hours = readings.assign(hour=readings["time"].dt.hour)[["hour", "celsius"]]
_, first = next(hours.iterrows())

print(first.dtype, first["hour"])
range(first["hour"])

float64 0.0


TypeError: 'numpy.float64' object cannot be interpreted as an integer

`iterrows` hands back each row as a Series, and a Series holds one type. The row held an integer and
a decimal, so pandas made both decimals, and the hour 0 became 0.0, which `range` refuses. A loop
with `itertuples` gets each row as a named tuple instead, which keeps each column's own type:

In [24]:
print(next(hours.itertuples(index=False)))

Pandas(hour=0, celsius=-3.6)


### No error, and float32 readings that match the originals only 7,067 times

In [25]:
as_float32 = readings["celsius"].astype("float32")

print("equal to the originals:", (as_float32 == readings["celsius"]).sum(), "of", readings["celsius"].count())
print(readings["celsius"].iloc[0], "->", float(as_float32.iloc[0]))

equal to the originals: 7067 of 35016
-3.6 -> -3.5999999046325684


`float32` stores -3.6 as -3.5999999046325684. Compared with a `float64` column, the `float32` values
are widened to `float64` first, and most of them no longer equal the originals, so a comparison or a
join on the readings quietly finds fewer rows. Keep `float64` for a column you compare or join on,
or round the `float32` values back to a tenth before comparing, which matches every reading:

In [26]:
print("equal after rounding:", (as_float32.astype(float).round(1) == readings["celsius"]).sum(), "of",
      readings["celsius"].count())

equal after rounding: 35016 of 35016


Last, the notebook is finished with its file, so this cell removes the scratch folder:

In [27]:
shutil.rmtree(SCRATCH)

print("scratch still there:", SCRATCH.exists())

scratch still there: False


## Recap

- Measure a table's memory with `memory_usage(deep=True)`, and read the `+` in `info`'s total -
  Store a column of repeated names as a `category`, and use `observed=True` when grouping by it -
  Shrink numbers with `pd.to_numeric(..., downcast=...)`, and keep `float64` where values are
  compared - Read only the columns you need, in the types you need, with `usecols` and `dtype`, and
  read a file in pieces with `chunksize` - Compute a column with one expression first, `apply` next,
  and `iterrows` last, and measure the difference on your own machine

## What is next

The **scikit-learn, Deep Dive** guide fits models to tables like these, and the library's front page
lists every guide.

---

&#8592; **Previous:** [Strings](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/17-strings.ipynb)  &nbsp;·&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
